# Same Anwser Anomaly

In [1]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt

## Import Data from csv

In [2]:
fca_question = pd.read_csv("../../../decoded_data/SJT/FactQuestionSJT.csv")

## Data Inspection

In [3]:
fca_question.head()

,QuestionKey,InstanceID,ItemID,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpent,Test,TestKey
0,1,1,1,5,4,2,181,SJT,160068
1,2,1,2,4,3,2,172,SJT,160068
2,3,1,3,4,2,3,136,SJT,160068
3,4,1,4,3,2,4,266,SJT,160068
4,5,1,5,2,5,1,198,SJT,160068


In [4]:
fca_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89622 entries, 0 to 89621
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   QuestionKey      89622 non-null  int64 
 1   InstanceID       89622 non-null  int64 
 2   ItemID           89622 non-null  int64 
 3   AnswerSequence1  89622 non-null  int64 
 4   AnswerSequence2  89622 non-null  int64 
 5   AnswerSequence3  89622 non-null  int64 
 6   TimeSpent        89622 non-null  int64 
 7   Test             89622 non-null  object
 8   TestKey          89622 non-null  int64 
dtypes: int64(8), object(1)
memory usage: 6.2+ MB


## Data Preparation

The only data cleaning that needs to happen here is getting rid of the coloms we won't need in this anomaly detection. There seem to be no empty fields or suspicious values to worry about. 

In [5]:
df = fca_question[["QuestionKey", "AnswerSequence1", "AnswerSequence2", "AnswerSequence3"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89622 entries, 0 to 89621
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   QuestionKey      89622 non-null  int64
 1   AnswerSequence1  89622 non-null  int64
 2   AnswerSequence2  89622 non-null  int64
 3   AnswerSequence3  89622 non-null  int64
dtypes: int64(4)
memory usage: 2.7 MB


## Data Labelling
We will now figure out on which tests the candidate filled out exactly the same answer for every question. 

In [6]:
same_answers = (df['AnswerSequence1'] == df['AnswerSequence2']) & (df['AnswerSequence2'] == df['AnswerSequence3'])
df['SameAnswers'] = False
df.loc[same_answers, 'SameAnswers'] = True

C:\Users\monad\AppData\Local\Temp\ipykernel_23516\4232101489.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['SameAnswers'] = False


In [7]:
df

,QuestionKey,AnswerSequence1,AnswerSequence2,AnswerSequence3,SameAnswers
0,1,5,4,2,False
1,2,4,3,2,False
2,3,4,2,3,False
3,4,3,2,4,False
4,5,2,5,1,False
...,...,...,...,...,...
89617,89618,0,0,0,True
89618,89619,0,0,0,True
89619,89620,0,0,0,True
89620,89621,0,0,0,True


## Exporting Data with Anomaly Check

In [8]:
df = df[["QuestionKey", "SameAnswers"]]
df.set_index("QuestionKey", inplace=True)
df

,SameAnswers
QuestionKey,
1,False
2,False
3,False
4,False
5,False
...,...
89618,True
89619,True
89620,True


In [9]:
df.to_csv("../csv/same_answers_question_checked.csv")